## 1. Session Overview

This notebook fine-tunes `meta-llama/Meta-Llama-3.1-8B-Instruct` using the MaxText framework on Kaggle TPU v5e.

- Objective: Run a minimal verification on TPU v5e, then proceed to fine-tuning.
- Evidence of Done: Successful `steps: 1` MaxText run and logs confirming TPU utilization.
- Artifacts: Config file, logs, and checkpoints saved to Kaggle outputs and/or GCS.

Preconditions:
- Kaggle accelerator set to TPU v5e.
- Internet enabled for cloning and dependency installs.
- Access to MaxText-compatible Llama 3.1 checkpoint via Kaggle Datasets.


## 2. Kaggle TPU v5e Environment Setup Plan

Steps in this session:
1. Verify TPU visibility and JAX version
2. Clone MaxText (main branch)
3. Install dependencies from `requirements.txt`
4. Prepare minimal `config.yaml` for verification run
5. Run a 1-step verification to confirm TPU v5e works

Notes:
- No substeps for now; each step maps to a single cell or small group of cells.
- We will capture logs and versions for reproducibility.


In [ ]:
# 3. Verify TPU visibility and JAX environment
import os, sys, platform, subprocess, json

print("Python:", sys.version)
print("Platform:", platform.platform())

# Kaggle TPU env vars
for key in ["TPU_NAME", "TPU_WORKER_ID", "TPU_CHIPS_PER_PROCESS", "TPU_MULTISLICE_CTRL_ADDRESS"]:
    if key in os.environ:
        print(f"{key}:", os.environ[key])

try:
    import jax
    import jaxlib
    import jax.numpy as jnp
    print("jax:", jax.__version__)
    print("jaxlib:", jaxlib.__version__)
    devices = jax.devices()
    print("Devices:")
    for d in devices:
        print(" -", d)
    print("Device count:", len(devices))
    x = jnp.ones((8, 8))
    y = jnp.dot(x, x).block_until_ready()
    print("JAX test dot result shape:", y.shape)
except Exception as e:
    print("[ERROR] JAX/TPU verification failed:", e)
    raise


## 4. Clone MaxText (main branch)

We will clone the official `google/maxtext` repository at the default `main` branch for the latest TPU v5e-compatible training scripts. Evidence of done: repository present in the working directory and HEAD commit printed.


In [ ]:
# 4. Clone MaxText (main)
set -e

echo "Cloning google/maxtext (main)..."
if [ ! -d "maxtext" ]; then
  git clone --depth=1 https://github.com/google/maxtext.git
else
  echo "Repository 'maxtext' already exists; skipping clone."
fi

cd maxtext
echo "Repo HEAD:" 
git log -1 --pretty=oneline || true

echo "Top-level files:" 
ls -1 | sed -n '1,50p'


## 5. Install dependencies from requirements.txt

Install Python dependencies required by MaxText. Kaggle TPU v5e includes a modern JAX stack; if a conflict arises, we will prefer the preinstalled JAX. Evidence of done: successful pip install and import checks.


In [ ]:
# 5. Install MaxText requirements
set -e

python -V
pip -V

echo "Installing MaxText requirements..."
pip install --no-input --no-cache-dir -r maxtext/requirements.txt

python - <<'PY'
import jax, jaxlib
print("jax:", jax.__version__)
print("jaxlib:", jaxlib.__version__)
print("JAX import successful after requirements install.")
PY
